# EGC KEGG Tree

Builds the KEGG annotation tree from `kegg_db.json` and writes one `.ko.json`
per sample to `DataStructures/Kegg/`.

**Run order:** top to bottom, selecting the KEGG INPUT cell (not the Taxonomy one).


In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r'C:\Users\sakoehn\Desktop\JGI\Notebooks')
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc


## 1. Load sample list and kegg_db


In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f'Loaded {len(samples)} samples')

with open(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json') as f:
    kegg_db = json.load(f)
print(f'kegg_db: {len(kegg_db):,} genes')


Loaded 128 samples
kegg_db: 36,803,002 genes


## 1.2. Load KEGG BRITE hierarchy (ko00001.keg)


In [3]:
with open(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\ko00001.keg') as f:
    keg_lines = f.readlines()

brite = {}          # {node_id: {name, depth, category, parent_id, parent_name}}
current_at_depth = {}

for line in keg_lines:
    if not any(line.startswith(c) for c in 'ABCDEFGHIJK'):
        continue
    level_char = line[0]
    depth      = ord(level_char) - ord('A') + 1
    rest       = line[1:].strip()
    if not rest:
        continue
    parts = rest.split(None, 1)
    if len(parts) < 2:
        continue
    node_id, name = parts[0], parts[1]
    parent        = current_at_depth.get(depth - 1)
    category      = name if depth == 1 else current_at_depth.get(1, {}).get('name')
    current_at_depth[depth] = {'id': node_id, 'name': name}
    for d in list(current_at_depth):
        if d > depth:
            del current_at_depth[d]
    brite[node_id] = {
        'name':        name,
        'depth':       depth,
        'category':    category,
        'parent_id':   parent['id']   if parent else None,
        'parent_name': parent['name'] if parent else None,
    }

print(f'BRITE: {len(brite):,} nodes')


BRITE: 28,893 nodes


## 2. Build KEGG tree

For each gene in `kegg_db`, strip the `KO:` prefix and walk up the BRITE
hierarchy from the KO to the root, registering scaffold counts at every level.


In [4]:
tax_tree = {}  # {node_id: {name, category, depth, parent_id, parent_name, timepoints}}

start = time.time()

for gene_id, ginfo in kegg_db.items():
    scaffold   = gene_id.rsplit('_', 2)[0]
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])

    for full_path, sample in zip(lookups, timepoints):
        # Strip 'KO:' prefix; handle semicolon-joined paths by taking first token
        ko_id = full_path.split(';')[0].strip()
        if ko_id.startswith('KO:'):
            ko_id = ko_id[3:]

        if ko_id not in brite:
            continue

        # Walk from the KO up to the A-level root, registering at every ancestor
        current = ko_id
        while current is not None:
            if current not in brite:
                break

            if current not in tax_tree:
                b = brite[current]
                tax_tree[current] = {
                    'name':        b['name'],
                    'category':    b['category'],
                    'depth':       b['depth'],
                    'parent_id':   b['parent_id'],
                    'parent_name': b['parent_name'],
                    'timepoints':  {},
                }

            node = tax_tree[current]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Only count this gene once per (KO node, sample) — a gene can
            # appear multiple times if it has multiple KO hits in kegg_db.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaffold] = tp['scaffolds'].get(scaffold, 0) + 1
            tp['genes'].add(gene_id)

            current = brite[current]['parent_id']

print(f'KEGG tree: {len(tax_tree):,} nodes  ({time.time() - start:.1f}s)')


KEGG tree: 12,699 nodes  (169.3s)


## 3. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a KEGG term (from this notebook's own `kegg_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_COG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a KEGG hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a KEGG-only subset of the shared total) can never exceed it.

In [5]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_COG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [6]:
def avg_gene_map(node_id):
    node = tax_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in tax_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 4. Write .ko.json files


In [7]:
base = Path(r'C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg')
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for node_id, node in tax_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_id] = {
            'name':        node['name'],
            'category':    node['category'],
            'depth':       node['depth'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp['egc'],
            'norm_egc':    tp['norm_egc'],
            'scaffolds':   tp.get('scaffolds', {}),
        }
    out_path = base / f'{s}.ko.json'
    with open(out_path, 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f'Wrote {len(samples)} .ko.json files to {base}')


Wrote 128 .ko.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg
